# CASMI 2026 — solo pipeline, offline inference

**Attach before running (Add Input):** the competition data and the private dataset built by
`solo/scripts/package_artifacts.py` (pool, fingerprints, library, S3 models, reranker, code). Internet **off**.

Everything heavy is precomputed; this notebook only scores `test.parquet`. No RDKit needed at run time.
Local check on the placeholder test: 400 molecules in 13 s, 1.9 GB RAM.

In [ ]:
import glob
import os
import subprocess
import sys
import time


def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    assert hits, f'not found: {pattern}'
    return hits

ART = os.path.dirname(find('pool.parquet')[0])
TEST = [p for p in find('test.parquet') if not p.startswith(ART)][0]
print('artifacts:', ART, os.listdir(ART))
print('test:', TEST)

In [ ]:
t0 = time.time()
subprocess.run([sys.executable, f'{ART}/code/predict.py', '--test', TEST, '--artifacts', ART,
                '--out', '/kaggle/working/submission.csv'], check=True)
print(f'done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
import pandas as pd

sub = pd.read_csv('/kaggle/working/submission.csv')
test_ids = pd.read_parquet(TEST, columns=['molecule_id']).molecule_id.unique()
assert set(sub.iloc[:, 0]) == set(test_ids), 'molecule ids differ from test'
print(sub.shape)
sub.head()